## Deploying Serverless APIs

Welcome back! In the previous lesson, you learned how AWS Lambda and API Gateway work together to create serverless APIs. Now, you are ready to take the next step: using the AWS Serverless Application Model (SAM) to define, test, and deploy your serverless applications.

AWS SAM is a framework that helps you manage serverless resources using simple configuration files. Instead of setting up each AWS service by hand, you can describe your entire application in a single file. This makes it easier to build, test, and deploy serverless APIs.

In this lesson, you will learn how to use AWS SAM to:

* Define your Lambda function and API endpoint in a `template.yaml` file
* Test your function locally with sample data
* Deploy your application to AWS and verify it works

By the end of this lesson, you will be able to take a Python function and turn it into a working API using AWS SAM.

---

## Quick Recap: Lambda Handlers and API Gateway

Before we dive in, let's quickly remind ourselves how Lambda and API Gateway work together.

* **Lambda Function:** This is your code that runs in the cloud. It takes an event (input), does some work, and returns a result.
* **API Gateway:** This is the front door to your Lambda. It lets users send HTTP requests (like POST or GET) to your function.

In the last lesson, you saw a basic Lambda handler in Python:

```python
def handler(event, context):
    # Your code here
    return {
        "statusCode": 200,
        "body": "Hello, world!"
    }
```

The `event` parameter contains the data sent to your function (like a JSON payload from an API request). The `context` parameter has information about the runtime, but for now, you can focus on `event`.

With this in mind, let's see how to use AWS SAM to manage and deploy this kind of function.

---

## Defining Your Application with `template.yaml`

The heart of AWS SAM is the `template.yaml` file. This file describes your serverless application in a way that AWS understands.

Let's build up a simple `template.yaml` step by step.

### 1. Start with the Template Header

Every SAM template starts with a header that tells AWS what kind of template it is:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example
```

* `AWSTemplateFormatVersion` and `Transform` are required for SAM templates.
* `Description` is optional but helpful.

### 2. Define the Lambda Function

Next, you define your Lambda function as a resource. Here's how you do it:

```yaml
Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
```

* `Resources` is where you list everything your app needs.
* `CalcFunction` is the name of your Lambda function.
* `Type: AWS::Serverless::Function` tells SAM this is a Lambda.
* `Properties` describe the function:
  * `CodeUri: .` means the code is in the current directory.
  * `Handler: main.handler` points to the Python file (`main.py`) and the function (`handler`).
  * `Runtime: python3.13` sets the Python version.
  * `Timeout` and `MemorySize` control how long and how much memory your function gets.

### 3. Connect API Gateway to Lambda

To make your function accessible over the web, you add an event trigger:

```yaml
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST
```

* `Events` lets you define triggers.
* Here, we set up an HTTP API that listens for POST requests at `/calculate`.

### 4. Add Outputs

Finally, you can add outputs to make it easy to find your API endpoint after deployment:

```yaml
Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/calculate"
```

This will print the API URL when you deploy.

### 5. Complete Template Example

Here's how the full `template.yaml` looks when you put it all together:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example

Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST

Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/calculate"
```

This template tells AWS to create a Lambda function, connect it to an API endpoint, and show you the URL after deployment.

---

## Testing Locally with Sample Event Data

Before you deploy your function to AWS, it's a good idea to test it locally. This helps you catch errors early.

### 1. Create a Sample Event

You can use a JSON file to simulate an API request. For example, create a file called `event.json`:

```json
{"amount": 150}
```

This file represents the data your API will receive.

### 2. Write the Lambda Handler

Here's a simple handler that reads the amount, calculates tax, and returns the result:

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

Let's break this down:

* `event.get("body")` gets the JSON body from the API request.
* `json.loads(...)` turns the JSON string into a Python dictionary.
* `amount = float(body.get("amount", 0))` reads the amount from the input.
* `tax` and `total` are calculated.
* The function returns a response with the results in JSON format.

### 3. Test the Function

On your own machine, you can use the AWS SAM CLI to run the function locally:

```shell
sam local invoke CalcFunction --event events/event.json
```

This command runs your function with the sample event.

**Expected Output:**

```text
{
    "statusCode": 200,
    "headers": {"Content-Type": "application/json"},
    "body": "{\"amount\": 150.0, \"tax\": 10.5, \"total\": 160.5}"
}
```

This shows that your function is working as expected before you deploy it.

---

## Deploying with SAM: Build, Deploy, and Verify

Once your function works locally, you are ready to deploy it to AWS.

### 1. Build the Application

First, you build your application so SAM can package it for deployment:

```shell
sam build
```

This command prepares your code and dependencies.

### 2. Deploy the Application

Next, you deploy your application to AWS:

```shell
sam deploy --stack-name tax-calculator-app --capabilities CAPABILITY_IAM --resolve-s3 --no-confirm-changeset --no-fail-on-empty-changeset
```

* `--stack-name` gives your deployment a name.
* `--capabilities CAPABILITY_IAM` allows SAM to create roles.
* `--resolve-s3` lets SAM manage storage for deployment.

### 3. Find Your API Endpoint

After deployment, SAM will print the API URL. For example:

```text
🌐 API URL: https://abc123.execute-api.us-east-1.amazonaws.com/calculate
```

### 4. Test the Deployed API

You can test your API using `curl`:

```shell
curl -X POST "https://abc123.execute-api.us-east-1.amazonaws.com/calculate" -H "Content-Type: application/json" -d '{"amount": 100}'
```

**Expected Output:**

```text
{"amount": 100.0, "tax": 7.0, "total": 107.0}
```

This confirms your function is live and working in the cloud.

---

## Summary and What's Next

In this lesson, you learned how to use AWS SAM to define, test, and deploy a serverless API. You saw how to:

* Describe your Lambda function and API endpoint in a `template.yaml` file
* Test your function locally with sample event data
* Deploy your application to AWS and verify it works

You are now ready to practice these steps yourself. In the next exercises, you will get hands-on experience defining, testing, and deploying your own serverless APIs using AWS SAM. Good luck, and have fun building!

## Fix the Broken API Configuration

Now that you understand how SAM templates work, it's time to get hands-on experience fixing a common configuration issue. You have been given a working Lambda function that calculates tax for a given amount, but there is a problem with the `template.yaml` configuration.

The API Gateway is not set up correctly to handle POST requests to the `/calculate` endpoint. Your job is to fix the `template.yaml` file so that the API responds properly to POST requests.

Look at the `Events` section under the Lambda function resource and make sure:

* The HTTP method is correct for receiving JSON data.
* The path matches what the Lambda handler expects.

Once you fix the template, you can test your changes locally using the provided event data to make sure everything works before deployment. This exercise will help you understand how SAM connects Lambda functions to API endpoints and prepare you for building your own serverless APIs.

> **Note:** the broken `Method: GET` shown below is a reconstruction based on the exercise description (GET requests don't carry a JSON body, which matches "the API Gateway is not set up correctly to handle POST requests"). Double-check it against the live CodeSignal exercise page and let me know if the actual bug differs.

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example

Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: GET

Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/calculate"
```

Here is the fixed `template.yaml` with the HTTP method corrected to `POST`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example

Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST

Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/calculate"
```

## Fix the Test Event Data

Excellent work on fixing the SAM template configuration! Now you're ready to tackle another important skill: creating proper test event data for local Lambda testing.

You have a working Lambda function that processes customer orders, but the current test event file has the wrong data structure. The function expects specific fields in a particular format, but the `events/event.json` file contains incorrect data that will cause errors or unexpected results.

Your task is to:

* Read the Lambda handler code to understand which data structure it expects
* Fix the `events/event.json` file with the correct JSON payload
* Test your changes using `sam local invoke` to verify that everything works

This exercise will teach you how to debug event data issues and create proper test files for your Lambda functions before deployment.

**events/event.json (broken)**

```json
{
  "customer_name": "John Smith",
  "product": "Laptop",
  "cost": "899.99"
}
```

**main.py**

```python
import json

def handler(event, context):
    try:
        # Parse the order data from the event
        order_data = event.get("order", {})
        customer_info = order_data.get("customer", {})
        items = order_data.get("items", [])
        
        # Validate required fields
        if not customer_info.get("name"):
            return {
                "statusCode": 400,
                "body": json.dumps({"error": "Customer name is required"})
            }
        
        if not items:
            return {
                "statusCode": 400,
                "body": json.dumps({"error": "Order must contain at least one item"})
            }
        
        # Calculate order totals
        subtotal = 0
        for item in items:
            price = float(item.get("price", 0))
            quantity = int(item.get("quantity", 1))
            subtotal += price * quantity
        
        # Calculate tax and shipping
        tax_rate = 0.08
        tax = round(subtotal * tax_rate, 2)
        
        shipping = 0
        if subtotal < 50:
            shipping = 9.99
        
        total = round(subtotal + tax + shipping, 2)
        
        # Build response
        response_data = {
            "customer": customer_info.get("name"),
            "order_summary": {
                "subtotal": subtotal,
                "tax": tax,
                "shipping": shipping,
                "total": total
            },
            "items_count": len(items)
        }
        
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(response_data)
        }
        
    except Exception as e:
        return {
            "statusCode": 500,
            "body": json.dumps({"error": str(e)})
        }
```

The handler reads `event["order"]["customer"]["name"]` and `event["order"]["items"]`, where each item needs a `price` and a `quantity` — none of which exist in the broken `event.json`. Here is the corrected `events/event.json`:

```json
{
  "order": {
    "customer": {
      "name": "John Smith"
    },
    "items": [
      {
        "product": "Laptop",
        "price": 899.99,
        "quantity": 1
      }
    ]
  }
}
```

`main.py` stays unchanged — only the test event data needed fixing.

## Complete the Lambda Function Properties

Perfect! You've learned how to fix API configurations and work with test event data. Now, let's focus on the foundation of every SAM template: properly defining Lambda function properties.

You have a working Python handler that performs math calculations and a complete test event file, but the `template.yaml` file is missing some key properties that tell AWS how to run your Lambda function. Your job is to complete the `template.yaml` file by adding the missing Lambda configuration.

Look at the `main.py` file to understand:

* What the handler function is called
* Which file contains your code

Then, fill in the missing properties in the template so SAM knows where to find your code, which function to call, and what Python version to use.

Once you complete the template, you can test your work by deploying with `sam deploy` and then invoking the deployed function to verify that everything connects properly. This exercise will give you hands-on experience with the core Lambda properties that every SAM template needs.

**template.yaml**

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Math calculation Lambda function

Resources:
  MathFunction:
    Type: AWS::Serverless::Function
    Properties:
      # TODO: Add CodeUri property to tell SAM where to find your code files
      # TODO: Add Handler property to specify which file and function to call
      # TODO: Add Runtime property to specify the Python version
      Timeout: 10
      MemorySize: 256

Outputs:
  FunctionName:
    Value: !Ref MathFunction
```

**main.py**

```python
import json

def handler(event, context):
    try:
        # Get the numbers from the event
        num1 = float(event.get("number1", 0))
        num2 = float(event.get("number2", 0))
        operation = event.get("operation", "add")
        
        # Perform the calculation
        if operation == "add":
            result = num1 + num2
        elif operation == "subtract":
            result = num1 - num2
        elif operation == "multiply":
            result = num1 * num2
        elif operation == "divide":
            if num2 == 0:
                return {
                    "statusCode": 400,
                    "body": json.dumps({"error": "Cannot divide by zero"})
                }
            result = num1 / num2
        else:
            return {
                "statusCode": 400,
                "body": json.dumps({"error": "Invalid operation"})
            }
        
        # Return the result
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({
                "number1": num1,
                "number2": num2,
                "operation": operation,
                "result": round(result, 2)
            })
        }
        
    except Exception as e:
        return {
            "statusCode": 500,
            "body": json.dumps({"error": str(e)})
        }
```

`main.py` defines `handler` inside `main.py`, so here is the completed `template.yaml` with the missing properties filled in:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Math calculation Lambda function

Resources:
  MathFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 10
      MemorySize: 256

Outputs:
  FunctionName:
    Value: !Ref MathFunction
```

## Add SAM Template Outputs Section

Nice work mastering the core Lambda properties! Now you're ready to learn about one of the most useful features of SAM templates: Outputs.

You have a complete, working serverless application with a Lambda function and a properly configured API Gateway endpoint. Everything works perfectly, but there's one problem: after you deploy the application, you have no easy way to find the API URL!

Your task is to add an `Outputs` section to the SAM template that will display the API Gateway URL after deployment. This is extremely helpful because AWS generates random URLs for your APIs, and without outputs, you'd have to dig through the AWS console to find them.

Look at the existing template and notice that it's missing the entire `Outputs` section. You need to add this section that:

* Creates an output called `ApiUrl`
* Uses CloudFormation intrinsic functions to build the complete API endpoint URL
* References the auto-generated API Gateway resource that SAM creates

Once you add the outputs, you can deploy your application, and SAM will show you the API URL right in the terminal, making it easy to test your deployed function.

> **Note:** the source pasted into this cell already included the finished `Outputs` block, which contradicts "notice that it's missing the entire Outputs section." The starter below is reconstructed by removing that block (as the description says it should be missing); the "completed" version below restores exactly the `Outputs` block that was already given. Double-check against the live CodeSignal page in case the real starter differs in another way.

**template.yaml (starter, missing Outputs)**

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example

Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST
```

**main.py**

```python
import json

def handler(event, context):
    try:
        # Parse the shipping request from the event body
        body = json.loads(event.get("body") or "{}")
        weight = float(body.get("weight", 0))
        distance = float(body.get("distance", 0))
        shipping_type = body.get("type", "standard")
        
        # Validate input
        if weight <= 0:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "Weight must be greater than 0"})
            }
        
        if distance <= 0:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "Distance must be greater than 0"})
            }
        
        # Calculate base shipping cost
        base_rate = 0.50  # $0.50 per pound
        distance_rate = 0.10  # $0.10 per mile
        
        base_cost = weight * base_rate
        distance_cost = distance * distance_rate
        
        # Apply shipping type multiplier
        if shipping_type == "express":
            multiplier = 2.0
        elif shipping_type == "overnight":
            multiplier = 3.5
        else:  # standard
            multiplier = 1.0
        
        total_cost = round((base_cost + distance_cost) * multiplier, 2)
        
        # Build response
        response_data = {
            "weight": weight,
            "distance": distance,
            "shipping_type": shipping_type,
            "base_cost": round(base_cost, 2),
            "distance_cost": round(distance_cost, 2),
            "total_cost": total_cost
        }
        
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(response_data)
        }
        
    except Exception as e:
        return {
            "statusCode": 500,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": str(e)})
        }
```

Here is the completed `template.yaml` with the `Outputs` section added:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Simple API Gateway -> Lambda example

Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST

Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/calculate"
```

`main.py` stays unchanged — only the `Outputs` section needed adding.

## Build Complete Serverless Application from Scratch

Fantastic! You've successfully worked through all the individual pieces of SAM templates. Now it's time to bring everything together and create a complete serverless application from the ground up.

You have been given a working Lambda function that handles discount calculations, but you need to build the entire SAM infrastructure around it. This is your chance to demonstrate mastery of the complete SAM development workflow.

Your tasks are:

* Create a complete `template.yaml` file with all necessary sections.
* Build an `events/event.json` file for local testing.
* Test your configuration using `sam local invoke`.

Read the `main.py` file carefully to understand what data structure the Lambda function expects. Then, create both the SAM template and test event file that will make this function work as a serverless API. This exercise will prove you understand how to build serverless applications from scratch using everything you've learned.

**template.yaml (TODO)**

```yaml
# TODO: Add the AWSTemplateFormatVersion (use '2010-09-09')
# TODO: Add the Transform for SAM (use AWS::Serverless-2016-10-31)
# TODO: Add a Description for your discount calculator API

# TODO: Add a Resources section with a Lambda function
# TODO: Name your function DiscountFunction and set Type to AWS::Serverless::Function
# TODO: Add Properties section with CodeUri, Handler, Runtime, Timeout, and MemorySize
# TODO: Add Events section with HttpApi event for POST requests to /discount path

# TODO: Add Outputs section that displays the API Gateway URL after deployment
```

**events/event.json (TODO)**

```json
// TODO: Create this file with test data that matches what the Lambda function expects
// TODO: Look at the main.py file to see what fields the function reads from the event body
// TODO: Include both "amount" and "discount_percentage" fields with appropriate test values
```

**main.py**

```python
import json

def handler(event, context):
    try:
        # Parse the discount request from the event body
        body = json.loads(event.get("body") or "{}")
        amount = float(body.get("amount", 0))
        discount_percentage = float(body.get("discount_percentage", 0))
        
        # Validate input
        if amount <= 0:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "Amount must be greater than 0"})
            }
        
        if discount_percentage < 0 or discount_percentage > 100:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "Discount percentage must be between 0 and 100"})
            }
        
        # Calculate discount and final price
        discount_amount = round(amount * discount_percentage / 100, 2)
        final_price = round(amount - discount_amount, 2)
        
        # Build response
        response_data = {
            "original_amount": amount,
            "discount_percentage": discount_percentage,
            "discount_amount": discount_amount,
            "final_price": final_price
        }
        
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(response_data)
        }
        
    except Exception as e:
        return {
            "statusCode": 500,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": str(e)})
        }
```

`main.py` reads `amount` and `discount_percentage` from the JSON-encoded `body` field of the event. Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Discount calculator API

Resources:
  DiscountFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /discount
            Method: POST

Outputs:
  ApiUrl:
    Value: !Sub "https://${ServerlessHttpApi}.execute-api.${AWS::Region}.amazonaws.com/discount"
```

And here is the completed `events/event.json`, matching the `body`-wrapped JSON string the handler expects:

```json
{
  "body": "{\"amount\": 200, \"discount_percentage\": 15}"
}
```